# 女性VTuber画像 → See-through PSD
ColabのL4 GPU・Python 3.12を使用します。上から順番に実行してください。会社PCへのインストールはありません。成果物は一時ディスクに生成されるため、最後にZIPを保存してください。


In [ ]:
# See-through公式READMEどおりの専用環境をColab内に作る
import sys, subprocess, json
from pathlib import Path
from datetime import datetime, timezone

ROOT = Path('/content/avatar-lab')
ROOT.mkdir(exist_ok=True)
RUN = ROOT / 'output' / datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
RUN.mkdir(parents=True, exist_ok=False)
LOG = RUN / 'setup.log'
SEE = ROOT / 'see-through'
VENV = ROOT / '.venv-seethrough'
PY = VENV / 'bin' / 'python'
SEE_COMMIT = 'a25a5498e031dc7fe01232d05dadaf67736277fb'

def run(args: list[str], cwd: Path | None = None) -> None:
    print('実行:', ' '.join(args), flush=True)
    result = subprocess.run(args, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    with LOG.open('a', encoding='utf-8') as handle:
        handle.write('\n$ ' + ' '.join(args) + '\n' + result.stdout)
    print(result.stdout[-4000:], flush=True)
    result.check_returncode()

assert sys.version_info[:2] == (3, 12)
run(['nvidia-smi'])
if not SEE.exists():
    run(['git', 'clone', '--no-checkout', 'https://github.com/shitagaki-lab/see-through.git', str(SEE)])
    run(['git', 'checkout', SEE_COMMIT], SEE)
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=SEE, text=True).strip() == SEE_COMMIT
for name in ('README.md', 'LICENSE'):
    (RUN / ('see-through-' + name)).write_text((SEE / name).read_text(), encoding='utf-8')
if not PY.exists() or subprocess.run([str(PY), '-m', 'pip', '--version'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT).returncode != 0:
    run([sys.executable, '-m', 'pip', 'install', 'virtualenv'])
    run([sys.executable, '-m', 'virtualenv', str(VENV)])
run([str(PY), '-m', 'pip', 'install', 'torch==2.8.0+cu128', 'torchvision==0.23.0+cu128', 'torchaudio==2.8.0+cu128', '--index-url', 'https://download.pytorch.org/whl/cu128'])
run([str(PY), '-m', 'pip', 'install', '-r', 'requirements.txt'], SEE)
if not (SEE / 'assets').exists():
    (SEE / 'assets').symlink_to(SEE / 'common' / 'assets', target_is_directory=True)
run([str(PY), '-c', "import torch; print(torch.__version__); print(torch.cuda.get_device_name()); assert torch.cuda.is_available(); assert torch.cuda.is_bf16_supported(), 'BF16対応GPUが必要です'"])
(RUN / 'pip-freeze.txt').write_text(subprocess.check_output([str(PY), '-m', 'pip', 'freeze'], text=True), encoding='utf-8')
print('セットアップ完了:', RUN)


In [ ]:
import urllib.request, hashlib

IMAGE_URL = 'https://raw.githubusercontent.com/naochan3/vtuber-dev/7a3705c294138b0179f39e633a4c6c07f53806f6/input/base.png'
EXPECTED_SHA256 = '103075e95db8d65f01581edde5e1b2f3d823c3fa2ce08d4cc8afa16e696f3da7'
with urllib.request.urlopen(IMAGE_URL, timeout=60) as response:
    image_bytes = response.read()
assert hashlib.sha256(image_bytes).hexdigest() == EXPECTED_SHA256, '画像の整合性エラー'
INPUT = RUN / 'input'
INPUT.mkdir(exist_ok=True)
SRC = INPUT / 'base.png'
assert not SRC.exists(), '既存入力は上書きしません'
SRC.write_bytes(image_bytes)
print('入力保存:', SRC)
print('SHA256:', EXPECTED_SHA256)


In [ ]:
import time

PSD_OUTPUT = RUN / 'psd'
assert not PSD_OUTPUT.exists(), '既存出力は上書きしません'
PSD_OUTPUT.mkdir()
INFER_LOG = RUN / 'inference.log'
command = [str(PY), 'inference/scripts/inference_psd.py', '--srcp', str(SRC), '--save_dir', str(PSD_OUTPUT), '--save_to_psd', '--tblr_split', '--group_offload']
print('PSD生成:', ' '.join(command), flush=True)
start = time.perf_counter()
with INFER_LOG.open('w', encoding='utf-8') as handle:
    process = subprocess.Popen(command, cwd=SEE, stdout=handle, stderr=subprocess.STDOUT, text=True)
    while process.poll() is None:
        time.sleep(20)
        print('経過秒:', round(time.perf_counter() - start), subprocess.check_output(['nvidia-smi', '--query-gpu=memory.used,utilization.gpu', '--format=csv,noheader'], text=True).strip(), flush=True)
print(INFER_LOG.read_text(encoding='utf-8')[-7000:])
assert process.returncode == 0, f'推論失敗: {INFER_LOG}'
metrics = {'elapsed_seconds': round(time.perf_counter() - start, 2), 'gpu': subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True).strip(), 'source_commit': SEE_COMMIT, 'input_sha256': EXPECTED_SHA256}
(RUN / 'cloud-metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
print('生成PSD:', [str(p.relative_to(RUN)) for p in PSD_OUTPUT.rglob('*.psd')])


In [ ]:
import base64, zipfile, urllib.request
from IPython.display import display, HTML, Image as DisplayImage

assert process.returncode == 0, 'PSD推論が成功してから保存してください'
PREPARE = RUN / 'prepare-tha3.py'
with urllib.request.urlopen('https://raw.githubusercontent.com/naochan3/vtuber-dev/f51ce31a8e328c68ab4ef49409318675742b602e/scripts/prepare-tha3.py', timeout=60) as response:
    PREPARE.write_bytes(response.read())
run([str(PY), str(PREPARE), '--input', str(SRC), '--output', str(INPUT / 'tha3.png'), '--head-box', '330', '15', '920', '585'])
psd_files = [p for p in PSD_OUTPUT.rglob('*.psd') if not p.stem.endswith('_depth')]
assert len(psd_files) == 1, f'PSD数が想定と異なります: {psd_files}'
PSD = psd_files[0]
inspect_code = '''import sys, json
from pathlib import Path
from psd_tools import PSDImage
from PIL import Image
psd = PSDImage.open(sys.argv[1])
items = [{'name': layer.name, 'kind': layer.kind, 'bbox': list(layer.bbox), 'visible': layer.visible} for layer in psd.descendants()]
manifest = {'size': list(psd.size), 'layer_count': len(items), 'layers': items}
Path(sys.argv[2]).write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(manifest, ensure_ascii=False))
psd.composite().save(sys.argv[3])
image = Image.open(sys.argv[4])
assert image.size == (512, 512) and image.mode == 'RGBA'
assert image.getchannel('A').getextrema()[0] == 0
print('THA3入力: 512x512 RGBA、透明背景OK')
'''
run([str(PY), '-c', inspect_code, str(PSD), str(RUN / 'layers.json'), str(RUN / 'psd-preview.png'), str(INPUT / 'tha3.png')])
QA = ROOT / '.venv-qa'
if not (QA / 'bin/python').exists():
    run([sys.executable, '-m', 'virtualenv', str(QA)])
QP = QA / 'bin/python'
run([str(QP), '-m', 'pip', 'install', 'ruff', 'mypy', 'types-Pillow'])
run([str(QP), '-m', 'ruff', 'check', str(PREPARE)])
run([str(QP), '-m', 'mypy', '--strict', str(PREPARE)])
NORMALIZER = RUN / 'normalize-psd.py'
with urllib.request.urlopen('https://raw.githubusercontent.com/naochan3/vtuber-dev/abc475d4ffe20da4e3cd0e0a96c8cc5ee80ee474/scripts/normalize-psd.py', timeout=60) as response:
    NORMALIZER.write_bytes(response.read())
run([str(QP), '-m', 'pip', 'install', 'psd-tools==1.14.2'])
run([str(QP), '-m', 'ruff', 'check', str(NORMALIZER)])
run([str(QP), '-m', 'mypy', '--strict', str(NORMALIZER)])
RIG_PSD = PSD_OUTPUT / 'base-rig.psd'
run([str(PY), str(NORMALIZER), '--input', str(PSD), '--output', str(RIG_PSD)])
run([str(PY), '-c', "import sys; from psd_tools import PSDImage; a=PSDImage.open(sys.argv[1]); b=PSDImage.open(sys.argv[2]); assert a.size==b.size and len(a)==len(b); assert a.composite().tobytes()==b.composite().tobytes(); print('レイヤー名以外の合成結果は同一です')", str(PSD), str(RIG_PSD)])
print('PSDプレビュー:')
display(DisplayImage(filename=str(RUN / 'psd-preview.png'), width=400))
ARCHIVE = RUN.parent / (RUN.name + '-avatar-home.zip')
assert not ARCHIVE.exists(), '既存ZIPは上書きしません'
with zipfile.ZipFile(ARCHIVE, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(RUN))
print('保存ZIP:', ARCHIVE.name, 'bytes:', ARCHIVE.stat().st_size)
assert ARCHIVE.stat().st_size < 100 * 1024 * 1024, '成果物が大きすぎます'
print('ZIP SHA256:', hashlib.sha256(ARCHIVE.read_bytes()).hexdigest())
from google.colab import files
files.download(str(ARCHIVE))
